# DX 704 Week 4 Project

This week's project will test the learning speed of linear contextual bandits compared to unoptimized approaches.
You will start with building a preference data set for evaluation, and then implement different variations of LinUCB and visualize how fast they learn the preferences.


The full project description, a template notebook and supporting code are available on GitHub: [Project 4 Materials](https://github.com/bu-cds-dx704/dx704-project-04).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Collect Rating Data

The file "recipes.tsv" in this repository has information about 100 recipes.
Make a new file "ratings.tsv" with two columns, recipe_slug (from recipes.tsv) and rating.
Populate the rating column with values between 0 and 1 where 0 is the worst and 1 is the best.
You can assign these ratings however you want within that range, but try to make it reflect a consistent set of preferences.
These could be your preferences, or a persona of your choosing (e.g. chocolate lover, bacon-obsessed, or sweet tooth).
Make sure that there are at least 10 ratings of zero and at least 10 ratings of one.


In [39]:
import pandas as pd

In [40]:
recipes = pd.read_csv('recipes.tsv', sep='\t')
recipes

,recipe_slug,recipe_title,recipe_introduction
0,falafel,Falafel,Falafel is a popular Middle Eastern dish made ...
1,spamburger,Spamburger,Spamburger is a type of hamburger that is made...
2,bacon-fried-rice,Bacon Fried Rice,Bacon fried rice is a savory and satisfying di...
3,chicken-fingers,Chicken Fingers,Chicken fingers are a popular dish made from c...
4,apple-crisp,Apple Crisp,Apple crisp is a classic dessert made with bak...
...,...,...,...
95,bacon-mac-and-cheese,Bacon Mac And Cheese,Bacon mac and cheese is a delicious and comfor...
96,chicken-alfredo-lasagna,Chicken Alfredo Lasagna,Chicken alfredo lasagna is a delicious twist o...
97,classic-beef-lasagna,Classic Beef Lasagna,Classic beef lasagna is a hearty and comfortin...
98,vegetarian-mushroom-lasagna,Vegetarian Mushroom Lasagna,Vegetarian mushroom lasagna is a delicious and...


Hint: You may find it more convenient to assign raw ratings from 1 to 5 and then remap them as follows.

`ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25`

In [41]:
# We will use a bacon lover personal mixed with my own preferences for non-bacon items
ratings = pd.DataFrame({
    'recipe_slug': recipes['recipe_slug'],
})
ratings['rating_raw'] = [
    3, 1, 5, 5, 5, 5, 5, 1, 2, 3, 5, 5, 5, 4, 4, 4, 5, 3, 4, 4, 2, 1, 5, 2, 5, 4, 5, 1, 1, 1, 1, 1, 2, 
    4, 5, 5, 1, 5, 4, 2, 5, 5, 3, 4, 3, 5, 1, 4, 4, 3, 5, 5, 4, 1, 2, 2, 4, 5, 5, 5, 5, 5, 5, 3, 3, 5, 5, 5, 5, 4, 3, 
    1, 1, 2, 2, 1, 2, 5, 2, 2, 1, 2, 3, 2, 1, 3, 3, 3, 5, 4, 3, 5, 5, 5, 5, 5, 5, 5, 2, 2
]
ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25
ratings.drop(columns='rating_raw', inplace=True)
ratings

,recipe_slug,rating
0,falafel,0.50
1,spamburger,0.00
2,bacon-fried-rice,1.00
3,chicken-fingers,1.00
4,apple-crisp,1.00
...,...,...
95,bacon-mac-and-cheese,1.00
96,chicken-alfredo-lasagna,1.00
97,classic-beef-lasagna,1.00
98,vegetarian-mushroom-lasagna,0.25


In [42]:
len(ratings[ratings['rating'] == 0])

16

In [43]:
len(ratings[ratings['rating'] == 1])

39

In [44]:
ratings.to_csv('ratings.tsv', sep='\t', index=False)

Submit "ratings.tsv" in Gradescope.

## Part 2: Construct Model Input

Use your file "ratings.tsv" combined with "recipe-tags.tsv" to create a new file "features.tsv" with a column recipe_slug, a column bias which is hard-coded to one, and a column for each tag that appears in "recipe-tags.tsv".
The tag column in this file should be a 0-1 encoding of the recipe tags for each recipe.
[Pandas reshaping function methods](https://pandas.pydata.org/docs/user_guide/reshaping.html) may be helpful.

The bias column will make later LinUCB calculations easier since it will just be another dimension.

Hint: For later modeling steps, it will be important to have the feature data (inputs) and the rating data (target outputs) in the same order.
It is highly recommended to make sure that "features.tsv" and "ratings.tsv" have the recipe slugs in the same order.

In [45]:
recipe_tags = pd.read_csv('recipe-tags.tsv', sep='\t')
recipe_tags

,recipe_slug,recipe_tag
0,spam-musubi,hawaiian
1,spam-musubi,nori
2,spam-musubi,onthego
3,spam-musubi,rice
4,spam-musubi,snack
...,...,...
747,bacon-souffle,breakfast
748,bacon-souffle,brunch
749,bacon-souffle,cheese
750,bacon-souffle,eggs


In [46]:
# Make one-hot columns for each tag
tags_one_hot = (
    pd.get_dummies(recipe_tags["recipe_tag"])
    .groupby(recipe_tags["recipe_slug"])
    .max()
    .astype(int)
)
tags_one_hot = tags_one_hot.reindex(recipes["recipe_slug"]).reset_index().rename(columns={"index": "recipe_slug"})
tags_one_hot

,recipe_slug,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,avocado,...,udonnoodles,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough
0,falafel,0,0,0,1,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,spamburger,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,bacon-fried-rice,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,chicken-fingers,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,apple-crisp,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,bacon-mac-and-cheese,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96,chicken-alfredo-lasagna,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
97,classic-beef-lasagna,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
98,vegetarian-mushroom-lasagna,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [47]:
features = pd.DataFrame({
    'recipe_slug': recipes['recipe_slug'],
    'bias': [1]*100
})
features

,recipe_slug,bias
0,falafel,1
1,spamburger,1
2,bacon-fried-rice,1
3,chicken-fingers,1
4,apple-crisp,1
...,...,...
95,bacon-mac-and-cheese,1
96,chicken-alfredo-lasagna,1
97,classic-beef-lasagna,1
98,vegetarian-mushroom-lasagna,1


In [48]:
features = features.merge(tags_one_hot, on="recipe_slug", how="left").fillna(0)

In [49]:
features

,recipe_slug,bias,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,...,udonnoodles,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough
0,falafel,1,0,0,0,1,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,spamburger,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,bacon-fried-rice,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,chicken-fingers,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,apple-crisp,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,bacon-mac-and-cheese,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96,chicken-alfredo-lasagna,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
97,classic-beef-lasagna,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
98,vegetarian-mushroom-lasagna,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [50]:
features.to_csv('features.tsv', sep='\t', index=False)

Submit "features.tsv" in Gradescope.

## Part 3: Linear Preference Model

Use your feature and rating files to build a ridge regression model with ridge regression's regularization parameter $\alpha$ set to 1.


Hint: If you are using scikit-learn modeling classes, you should use `fit_intercept=False` since that intercept value will be redundant with the bias coefficient.

Hint: The estimate component of the bounds should match the previous estimate, so you should be able to just focus on the variance component of the bounds now.

In [51]:
from sklearn.linear_model import Ridge

X = features.drop(columns=['recipe_slug'])
y = ratings['rating']

ridge = Ridge(fit_intercept=False)
ridge.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",False
,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga'

Save the coefficients of this model in a file "model.tsv" with columns "recipe_tag" and "coefficient".
Do not add anything for the `intercept_` attribute of a scikit-learn model; this will be covered by the coefficient for the bias column added in part 2.

In [52]:
model = pd.DataFrame({
    'recipe_tag': ridge.feature_names_in_,
    'coefficient': ridge.coef_
})
model

,recipe_tag,coefficient
0,bias,0.394171
1,alfredo,0.141692
2,almond,-0.234453
3,american,0.024178
4,appetizer,0.097509
...,...,...
292,vegetarian,-0.122252
293,warm,0.049585
294,whippedcream,0.015773
295,winter,0.037828


In [53]:
model.to_csv('model.tsv', sep='\t', index=False)

Submit "model.tsv" in Gradescope.

## Part 4: Recipe Estimates

Use the recipe model to estimate the score of every recipe.
Save these estimates to a file "estimates.tsv" with columns recipe_slug and score_estimate.

In [54]:
y_pred = ridge.predict(X)

In [55]:
estimates = pd.DataFrame({
    'recipe_slug': features['recipe_slug'],
    'score_estimate': y_pred,
})
estimates

,recipe_slug,score_estimate
0,falafel,0.504622
1,spamburger,0.052254
2,bacon-fried-rice,0.976908
3,chicken-fingers,0.911242
4,apple-crisp,0.982725
...,...,...
95,bacon-mac-and-cheese,0.974692
96,chicken-alfredo-lasagna,0.858308
97,classic-beef-lasagna,0.859410
98,vegetarian-mushroom-lasagna,0.357654


Submit "estimates.tsv" in Gradescope.

In [56]:
estimates.to_csv('estimates.tsv', sep='\t', index=False)

## Part 5: LinUCB Bounds

Calculate the upper bounds of LinUCB using data corresponding to trying every recipe once and receiving the rating in "ratings.tsv" as the reward.
Keep the ridge regression regularization parameter at 1, and set LinUCB's $\alpha$ parameter to 2.
Save these upper bounds to a file "bounds.tsv" with columns recipe_slug and score_bound.

In [57]:
import numpy as np

In [58]:
[np.identity(3) for _ in range(2)]

[array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]),
 array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])]

In [59]:
[np.zeros(3) for _ in range(2)]

[array([0., 0., 0.]), array([0., 0., 0.])]

In [60]:
class LinUCB():
    # I am using the original disjoint version of LinUCB but there is also hybrid approach https://dl.acm.org/doi/10.1145/1772690.1772758
    # Note I ran into one issue in the grader where the Part 5 bounds were not matching the grader. I figured out it is due to the
    # previous implementation using a separate matrix per arm. By passing shared i can now use a single matrix shared across all arms which
    # will match the grader.
    def __init__(self, n_arms, n_features, alpha=2, shared=False):
        self.n_arms = n_arms
        self.n_features = n_features
        # alpha gets a default of 2 since it is required for this problem specifically
        self.alpha = alpha
        # shared flag is for the issue i mention above
        self.shared = shared

        if self.shared:
            # d-dimensional identity matrix. one shared for all arms
            self.A = np.identity(n_features)
            # d-dimensional zero vector. one shared for all arms
            self.b = np.zeros((n_features, 1))
        else:
            # d is just the num of features in the context. repeat for each arm a
            # d-dimensional identity matrix
            self.A = [np.identity(n_features) for _ in range(n_arms)]
            # d-dimensional zero vector
            self.b = [np.zeros((n_features, 1)) for _ in range(n_arms)]

    def score_bound(self, arm_idx, context):
        """Normally we would inline this method in choose_arm but it is directly needed for prob 5"""
        x = context.reshape(-1, 1)

        # line 8 of algo: θhat = A_a^-1 b_a
        if self.shared:
            A_inv = np.linalg.inv(self.A)
            theta_hat = A_inv @ self.b
        else:
            A_inv = np.linalg.inv(self.A[arm_idx])
            theta_hat = A_inv @ self.b[arm_idx]

        # line 9 of algo: p_t,a = (θhat_a)^t x_t,a + α*sqrt((x_t,a)^T (A_a)^-1 x_t,a)
        # it is basically the payoff we expect + uncertainty addition
        bound = (theta_hat.T @ x) + (self.alpha * np.sqrt(x.T @ A_inv @ x))
        return float(np.squeeze(bound))

    def update(self, arm_idx, context, reward):
        x = context.reshape(-1, 1)

        # line 12 of algo: A_a = A_a + x_t * x_t^T
        # line 13 of algo: b_a = b_a + r_t * x_t
        if self.shared:
            self.A = self.A + x @ x.T
            self.b = self.b + reward * x
        else:
            self.A[arm_idx] += x @ x.T
            self.b[arm_idx] += reward * x

In [61]:
X.shape

(100, 297)

In [62]:
reward_by_slug = ratings.set_index('recipe_slug')['rating'].to_dict()
X = features.drop(columns=['recipe_slug']).to_numpy()

linucb = LinUCB(n_arms=len(features), n_features=X.shape[1], alpha=2, shared=True)

for arm_idx, recipe_slug in enumerate(features['recipe_slug']):
    context = X[arm_idx]
    reward = reward_by_slug[recipe_slug]
    linucb.update(arm_idx, context, reward)

bounds = pd.DataFrame({
    'recipe_slug': features['recipe_slug'],
    'score_bound': [linucb.score_bound(i, X[i]) for i in range(len(features))]
})
bounds

,recipe_slug,score_bound
0,falafel,2.270877
1,spamburger,1.946590
2,bacon-fried-rice,2.843065
3,chicken-fingers,2.705358
4,apple-crisp,2.759067
...,...,...
95,bacon-mac-and-cheese,2.796246
96,chicken-alfredo-lasagna,2.626381
97,classic-beef-lasagna,2.569903
98,vegetarian-mushroom-lasagna,1.947774


In [63]:
bounds.to_csv('bounds.tsv', sep='\t', index=False)
bounds

,recipe_slug,score_bound
0,falafel,2.270877
1,spamburger,1.946590
2,bacon-fried-rice,2.843065
3,chicken-fingers,2.705358
4,apple-crisp,2.759067
...,...,...
95,bacon-mac-and-cheese,2.796246
96,chicken-alfredo-lasagna,2.626381
97,classic-beef-lasagna,2.569903
98,vegetarian-mushroom-lasagna,1.947774


Submit "bounds.tsv" in Gradescope.

## Part 6: Make Online Recommendations

Implement LinUCB to make 100 recommendations starting with no data and using the same parameters as in part 5.
One recommendation should be made at a time and you can break ties arbitrarily.
After each recommendation, use the rating from part 1 as the reward to update the LinUCB data.
Record the recommendations made in a file "recommendations.tsv" with columns "recipe_slug", "score_bound", and "reward".
The rows in this file should be in the same order as the recommendations were made.

Hint: do not remove recipes after each recommendation.
Repeating recommendations is expected.

In [68]:
linucb = LinUCB(n_arms=len(features), n_features=X.shape[1])
recommendations = []

for _ in range(100):
    best_score = -np.inf
    best_arm = None
    print(f"Iteration: {_}")

    for arm_idx in range(len(features)):
        score_bound = linucb.score_bound(arm_idx, X[arm_idx])

        # Break ties arbitrarily. We will just only take strictly better ones. So LIFO
        if score_bound > best_score:
            best_score = score_bound
            best_arm = arm_idx

    recipe_slug = features.iloc[best_arm]['recipe_slug']
    reward = reward_by_slug[recipe_slug]

    recommendations.append({
        'recipe_slug': recipe_slug,
        'score_bound': best_score,
        'reward': reward,
    })

    linucb.update(best_arm, X[best_arm], reward)

recommendations = pd.DataFrame(recommendations)

Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8
Iteration: 9
Iteration: 10
Iteration: 11
Iteration: 12
Iteration: 13
Iteration: 14
Iteration: 15
Iteration: 16
Iteration: 17
Iteration: 18
Iteration: 19
Iteration: 20
Iteration: 21
Iteration: 22
Iteration: 23
Iteration: 24
Iteration: 25
Iteration: 26
Iteration: 27
Iteration: 28
Iteration: 29
Iteration: 30
Iteration: 31
Iteration: 32
Iteration: 33
Iteration: 34
Iteration: 35
Iteration: 36
Iteration: 37
Iteration: 38
Iteration: 39
Iteration: 40
Iteration: 41
Iteration: 42
Iteration: 43
Iteration: 44
Iteration: 45
Iteration: 46
Iteration: 47
Iteration: 48
Iteration: 49
Iteration: 50
Iteration: 51
Iteration: 52
Iteration: 53
Iteration: 54
Iteration: 55
Iteration: 56
Iteration: 57
Iteration: 58
Iteration: 59
Iteration: 60
Iteration: 61
Iteration: 62
Iteration: 63
Iteration: 64
Iteration: 65
Iteration: 66
Iteration: 67
Iteration: 68
Iteration: 69
Iteration: 70
Iteration: 71
It

In [69]:
recommendations

,recipe_slug,score_bound,reward
0,apple-crumble,7.483315,1.00
1,ramen,7.211103,0.50
2,ma-la-chicken,7.211103,0.00
3,quesadillas,7.211103,0.75
4,spamburger,6.928203,0.00
...,...,...,...
95,tempura-udon,4.898979,0.75
96,peach-cobbler,4.898979,1.00
97,blueberry-crumble,4.898979,0.50
98,chiles-rellenos,4.898979,0.25


In [70]:
recommendations.to_csv('recommendations.tsv', sep='\t', index=False)
recommendations

,recipe_slug,score_bound,reward
0,apple-crumble,7.483315,1.00
1,ramen,7.211103,0.50
2,ma-la-chicken,7.211103,0.00
3,quesadillas,7.211103,0.75
4,spamburger,6.928203,0.00
...,...,...,...
95,tempura-udon,4.898979,0.75
96,peach-cobbler,4.898979,1.00
97,blueberry-crumble,4.898979,0.50
98,chiles-rellenos,4.898979,0.25


Submit "recommendations.tsv" in Gradescope.

## Part 7: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgements are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.


Submit "project.ipynb" in Gradescope.